# Baseline eval: prompted Qwen3.5-2B on the 100 golden cases

Serves the base Qwen3.5-2B with vLLM (the setup that passed the GPU risk tests) and runs `evals/run_eval.py`:
every golden case through the agent loop (prompt v0, the six rough tools, database on Neon), every trace saved,
scored, and a per-category report written. Then the sampling comparison: 30 cases (5 per category) with the
model card's settings, against the same 30 cases at temperature 0.2.

**Before running (Kaggle):** Settings → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets → **`NEON_URL`** ticked
(`HF_TOKEN` optional). Set `RUN_DATE` in cell 1, then **Run All** (about 30 minutes: 10 to start vLLM, the rest to run).
**After:** Output → download **`baseline-runs.zip`** and unzip it into the repo's `evals/` folder (it contains `runs/`).


In [ ]:
# 1. Settings and secrets. The run name carries the date; nothing here reads the clock.
RUN_DATE = "2026-11-22"          # today's date, written by hand
MODEL = "Qwen/Qwen3.5-2B"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["DB_URL"] = secrets.get_secret("NEON_URL")
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass
print("DB host:", os.environ["DB_URL"].split("@")[-1].split("/")[0])


In [ ]:
# 2. Clone the repo and install what the eval needs in this kernel.
import subprocess, sys
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
%pip install -q openai "psycopg[binary]" pydantic
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python evals/validate.py     # the golden set must match the database the eval will query


In [ ]:
# 3. vLLM in its own environment, pinned to the versions that passed the risk tests.
!uv --version || pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python -r requirements-serve.txt


In [ ]:
# 4. Start vLLM (T4 libcuda fix first) and wait until it answers.
import time, urllib.request
FLAGS = ("--dtype half --max-model-len 8192 --language-model-only "
         "--enable-auto-tool-choice --tool-call-parser qwen3_coder --port 8000")
log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(["bash", "-c", f"source serving/t4_libcuda_fix.sh && "
                           f"exec /tmp/vllm-env/bin/vllm serve {MODEL} {FLAGS}"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1500:
            print(open("/kaggle/working/vllm.log").read()[-4000:])
            raise RuntimeError("vLLM did not start; full log in /kaggle/working/vllm.log")
        time.sleep(10)
print(f"vLLM up after {time.time() - t0:.0f}s")


In [ ]:
# 5. The floor: all 100 cases at temperature 0.2. Re-running this cell resumes where it stopped.
FLOOR = f"{RUN_DATE}_floor"
!python evals/run_eval.py --run {FLOOR} --system floor --provider openai \
    --base-url http://localhost:8000/v1 --model {MODEL} --temperature 0.2 \
    --notes "Kaggle T4, vLLM 0.34.0"


In [ ]:
# 6. Sampling comparison: 30 cases (the first 5 of each category) with the model card's settings
#    (temperature 1.0, top_p 1.0, top_k 20, presence_penalty 2.0). The same 30 at 0.2 come from cell 5.
import json
from collections import defaultdict
per_cat = defaultdict(list)
for line in open(f"evals/runs/{FLOOR}/traces.jsonl"):
    t = json.loads(line)
    per_cat[t["category"]].append(t["id"])
SAMPLE30 = [i for ids in per_cat.values() for i in sorted(ids)[:5]]
CARD = f"{RUN_DATE}_floor-model-card"
!python evals/run_eval.py --run {CARD} --system floor --provider openai \
    --base-url http://localhost:8000/v1 --model {MODEL} --temperature 1.0 --sampling model-card \
    --ids {" ".join(SAMPLE30)} --notes "sampling comparison, 30 cases"


In [ ]:
# 7. The same 30 cases side by side: temperature 0.2 vs the model card.
from evals.run_eval import METRICS
s02 = json.load(open(f"evals/runs/{FLOOR}/scores.json"))["scores"]
scard = json.load(open(f"evals/runs/{CARD}/scores.json"))["scores"]
print(f"{'metric':<12}{'temp 0.2':>10}{'model card':>12}   (same 30 cases)")
for m in METRICS:
    print(f"{m:<12}{sum(s02[i][m] for i in SAMPLE30):>7}/30{sum(scard[i][m] for i in SAMPLE30):>9}/30")


In [ ]:
# 8. Package the runs for download: Output -> baseline-runs.zip, unzip into the repo's evals/ folder.
!cd evals && zip -qr /kaggle/working/baseline-runs.zip runs/{FLOOR} runs/{CARD}
!ls -la /kaggle/working/baseline-runs.zip
print(open(f"evals/runs/{FLOOR}/report.md").read())
